In [22]:
# ADMS 4509 A, F26
# Dr. Hamid Arian
# Term 2 Group Project
# Part 6: EVT-based VaR and ES
# Zakariya Asif

In [6]:
# Inspecting whether working directory is the project root or ../src
from pathlib import Path
import pandas as pd

print("Notebook working directory:", Path.cwd())

Notebook working directory: C:\Users\zakar\Jupyter_Projects\4509_EVT\src


In [7]:
# Checking if I have openpyxl in my environment
import openpyxl
print(openpyxl.__version__)

3.1.5


In [5]:
# Working directory is ../4509_EVT/src
project_root = Path.cwd().parent

In [11]:
# Attempting to read the .xlsx file provided by Rowan
data_path = (
    project_root
    / "data"
    / "5 Year Historical Stock Price Data - ADMS4509.xlsx"
)

if not data_path.is_file():
    raise FileNotFoundError(f"Worbook not found: {data_path}")

tickers = ["AMZN", "WMT", "JPM", "JNJ", "MSFT"]

raw_returns = pd.read_excel(
    data_path,
    sheet_name="Daily Returns",
    header=1, # The column names are on the second Excel row
    usecols=["Date", *tickers],
    engine="openpyxl",
)

raw_returns.head()

,Date,AMZN,WMT,JPM,JNJ,MSFT
0,2026-11-16,-0.033984,0.002495,-0.023321,-0.011262,-0.011307
1,2026-11-13,-0.002921,0.014944,0.009782,0.007835,0.001681
2,2026-11-12,0.004760,0.004129,-0.001661,0.011030,0.003582
3,2026-11-11,-0.017504,0.000955,0.003851,-0.001319,-0.022822
4,2026-11-10,0.014653,-0.000876,-0.000029,-0.004726,-0.022294


In [15]:
# Validating prior to cleansing
## Keeping the imported data separate from the cleaned, in order to inspect what changes.
## The original .xlsx workbook has a dated row with no returns at the beginning of the price history; expected as a return needs a previous price
## Investigating unexpected missing values or duplicate dates
## I'll make sure I don't replace missing returns with zero or forward-fill prices without investigating the cause, so as to not distrot the tail down the line.
returns = raw_returns.copy()
returns["Date"] = pd.to_datetime(returns["Date"], errors="raise")

print("Rows:", len(returns))
print("Date range:", returns["Date"].min(), "to", returns["Date"].max())
print("\nMissing values:")
print(returns.isna().sum())
print("\nDuplicate dates:", returns["Date"].duplicated().sum())
print("\nData types:")
print(returns.dtypes)

Rows: 1255
Date range: 2021-11-16 00:00:00 to 2026-11-16 00:00:00

Missing values:
Date    0
AMZN    1
WMT     1
JPM     1
JNJ     1
MSFT    1
dtype: int64

Duplicate dates: 0

Data types:
Date    datetime64[ns]
AMZN           float64
WMT            float64
JPM            float64
JNJ            float64
MSFT           float64
dtype: object


In [18]:
# One missing value for each stock pick (expected, as outlined above), zero duplicate dates.
# Sorting chronologically and removing rows that have no usable returns.
returns = returns.dropna(subset=["Date"])
returns = returns.dropna(subset=tickers, how="all")
returns = returns.sort_values("Date").reset_index(drop=True)

if returns["Date"].duplicated().any():
    raise ValueError("Duplicate dates found; investigate before proceeding.")
if returns[tickers].isna().any().any():
    raise ValueError("Some stock returns are missing; investigate before proceeding.")

returns.head()

,Date,AMZN,WMT,JPM,JNJ,MSFT
0,2021-11-17,0.002313,-0.008738,-0.005423,0.003739,0.000675
1,2021-11-18,0.040585,0.008738,-0.008711,-0.005366,0.006328
2,2021-11-19,-0.005263,-0.005368,-0.013140,0.002969,0.005349
3,2021-11-22,-0.028695,0.016459,0.021093,-0.013217,-0.009595
4,2021-11-23,0.002069,0.007254,0.023622,0.006133,-0.006355


In [23]:
# Seaprating the estimation period.
## Assuming position was taken on Oct 8, 2026, at market open - only returns preceding this date should be retained.
investment_date = pd.Timestamp("2026-10-08")

estimation_returns = returns.loc[
    returns["Date"] < investment_date
    ].copy()

print(
    f"{len(estimation_returns)} observations: "
    f"{estimation_returns['Date'].min():%Y-%m-%d} to "
    f"{estimation_returns['Date'].max():%Y-%m-%d}"
)

1226 observations: 2021-11-17 to 2026-10-07


In [24]:
# Original workbook yielded approximately 1,226 complete observations; observation count output above matches the original workbook.
# Preserving the return convention.
## The original workbook's daily values are log returns calculated from adjusted close prices. Before weighting them, I'll convert them to simple returns and subsequently calculate the hypothetical fixed-weight portfolio's daily loss.
import numpy as np

weights = pd.Series(0.20, index=tickers)

simple_returns = np.expm1(estimation_returns[tickers])
estimation_returns["portfolio_return"] = simple_returns @ weights
estimation_returns["portfolio_loss"] = -estimation_returns["portfolio_return"]

estimation_returns[["Date", "portfolio_return", "portfolio_loss"]].head()

,Date,portfolio_return,portfolio_loss
0,2021-11-17,-0.001474,0.001474
1,2021-11-18,0.008504,-0.008504
2,2021-11-19,-0.003064,0.003064
3,2021-11-22,-0.002611,0.002611
4,2021-11-23,0.006614,-0.006614
